# Profitability, sizing and paper-P&L analysis

This notebook answers four practical questions:

1. How often did the monitor find a feasible trade?
2. Which direction and ETH size produced the best estimated net P&L?
3. Where did the gross price edge go after Binance fees, Ethereum gas and rebalancing?
4. What did the simulated inventory ledger record as cumulative paper P&L?

All values are research marks from stored observations—not guaranteed fills or realized profit. The monitor uses Uniswap QuoterV2 and Binance depth when it creates these rows; this notebook does not re-quote historical markets.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")


def find_repo_root(start=Path.cwd()):
    # Work whether Jupyter starts in the repository root or notebooks/.
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "scr").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory")


ROOT = find_repo_root()
DB_PATH = Path(os.environ.get("ARB_DB_PATH", ROOT / "data" / "observations.sqlite3"))
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"No research database at {DB_PATH}. Run `python -m scr --config config.toml monitor` first, "
        "or set ARB_DB_PATH to another observations.sqlite3 file."
    )

# Read-only URI prevents a notebook from accidentally changing the research dataset.
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    connection,
)["name"].tolist()
print(f"Database: {DB_PATH}")
print("Tables:", ", ".join(tables))

## Load and validate the dataset

In [ ]:
observations = pd.read_sql_query("SELECT * FROM observations ORDER BY id", connection)
candidates = pd.read_sql_query("SELECT * FROM candidates ORDER BY observation_id, id", connection)
paper_trades = pd.read_sql_query("SELECT * FROM paper_trades ORDER BY session_id, id", connection)

numeric_observation_columns = [
    "market_age_ms", "binance_best_bid", "binance_best_ask", "uniswap_price", "binance_taker_fee"
]
numeric_candidate_columns = [
    "quantity_eth", "gross_buy_usdc", "gross_sell_usdc", "exchange_fee_usdc",
    "gas_cost_usdc", "gas_cost_eth", "rebalancing_cost_usdc", "net_profit_usdc", "profit_bps",
]
for column in numeric_observation_columns:
    observations[column] = pd.to_numeric(observations[column], errors="coerce")
for column in numeric_candidate_columns:
    candidates[column] = pd.to_numeric(candidates[column], errors="coerce")

observations["observed_at"] = pd.to_datetime(observations["observed_at"], utc=True, errors="coerce")
observations["cex_mid"] = (observations["binance_best_bid"] + observations["binance_best_ask"]) / 2
observations["cex_spread_usdc"] = observations["binance_best_ask"] - observations["binance_best_bid"]
observations["cex_spread_bps"] = observations["cex_spread_usdc"] / observations["cex_mid"] * 10_000
observations["dex_minus_cex_usdc"] = observations["uniswap_price"] - observations["cex_mid"]
observations["dex_minus_cex_bps"] = observations["dex_minus_cex_usdc"] / observations["cex_mid"] * 10_000

selected = candidates[candidates["selected"].eq(1)].copy()
selected = selected.merge(
    observations[["id", "observed_at", "cex_mid", "dex_minus_cex_usdc", "dex_minus_cex_bps", "stale"]],
    left_on="observation_id", right_on="id", how="left", suffixes=("", "_observation"),
)

summary = pd.Series({
    "observations": len(observations),
    "candidate quotes": len(candidates),
    "selected opportunities": len(selected),
    "paper trades": len(paper_trades),
    "first observation": observations["observed_at"].min(),
    "last observation": observations["observed_at"].max(),
    "stale observations": int(observations["stale"].sum()) if len(observations) else 0,
    "observations with errors": int(observations["error"].notna().sum()) if len(observations) else 0,
})
summary.to_frame("value")

A `selected` row means it was the optimizer's best candidate and cleared its configured constraints. It is still only an opportunity mark. Check stale/error counts before interpreting profitability.

## Market basis and observation quality

In [ ]:
if observations.empty:
    print("No observations yet. Run the monitor before analysing profitability.")
else:
    display(observations[[
        "market_age_ms", "cex_spread_usdc", "cex_spread_bps",
        "dex_minus_cex_usdc", "dex_minus_cex_bps",
    ]].describe(percentiles=[.05, .25, .5, .75, .95]).T)

    axes = observations.plot(
        x="observed_at", y=["cex_mid", "uniswap_price"], figsize=(12, 4),
        title="Observed ETH prices: Binance mid vs Uniswap spot",
    )
    axes.set_ylabel("USDC per ETH")
    plt.show()

    axes = observations.plot(
        x="observed_at", y="dex_minus_cex_usdc", figsize=(12, 3),
        title="Raw Uniswap minus Binance-mid basis (before costs)", legend=False,
    )
    axes.axhline(0, color="black", linewidth=1)
    axes.set_ylabel("USDC per ETH")
    plt.show()

## Selected opportunity frequency, direction and size

In [ ]:
if selected.empty:
    print("No selected opportunities. This can be a valid result when fees and gas exceed the raw spread.")
else:
    elapsed_hours = max(
        (observations["observed_at"].max() - observations["observed_at"].min()).total_seconds() / 3600,
        1 / 3600,
    )
    opportunity_summary = pd.Series({
        "selected opportunities": len(selected),
        "share of observations": len(selected) / max(len(observations), 1),
        "opportunities per observed hour": len(selected) / elapsed_hours,
        "median size ETH": selected["quantity_eth"].median(),
        "median net profit USDC": selected["net_profit_usdc"].median(),
        "total marked profit USDC (not additive fills)": selected["net_profit_usdc"].sum(),
    })
    display(opportunity_summary.to_frame("value"))
    display(selected.groupby("direction").agg(
        opportunities=("id", "size"),
        median_size_eth=("quantity_eth", "median"),
        median_profit_usdc=("net_profit_usdc", "median"),
        p95_profit_usdc=("net_profit_usdc", lambda values: values.quantile(.95)),
    ))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for direction, group in selected.groupby("direction"):
        axes[0].scatter(group["quantity_eth"], group["net_profit_usdc"], s=18, alpha=.6, label=direction)
        axes[1].plot(group["observed_at"], group["net_profit_usdc"], ".", alpha=.6, label=direction)
    axes[0].set(xlabel="Selected quantity (ETH)", ylabel="Estimated net profit (USDC)", title="Profit vs optimal size")
    axes[1].set(xlabel="Time", ylabel="Estimated net profit (USDC)", title="Selected opportunities over time")
    axes[0].legend(); axes[1].legend(); fig.autofmt_xdate(); plt.show()

The sum of selected marks is **not** a trading return: adjacent observations can describe the same short-lived opportunity. Use `paper_trades` for the system's simulated fill ledger and the latency notebook for opportunity survival.

## Cost decomposition and break-even spread

In [ ]:
analysis = selected.copy()
if analysis.empty:
    print("Cost decomposition requires at least one selected opportunity.")
else:
    analysis["gross_edge_usdc"] = analysis["gross_sell_usdc"] - analysis["gross_buy_usdc"]
    analysis["total_cost_usdc"] = (
        analysis["exchange_fee_usdc"] + analysis["gas_cost_usdc"] + analysis["rebalancing_cost_usdc"]
    )
    analysis["break_even_edge_per_eth_usdc"] = analysis["total_cost_usdc"] / analysis["quantity_eth"]
    analysis["gross_edge_per_eth_usdc"] = analysis["gross_edge_usdc"] / analysis["quantity_eth"]
    analysis["gas_per_eth_usdc"] = analysis["gas_cost_usdc"] / analysis["quantity_eth"]
    analysis["exchange_fee_per_eth_usdc"] = analysis["exchange_fee_usdc"] / analysis["quantity_eth"]

    display(analysis[[
        "gross_edge_usdc", "exchange_fee_usdc", "gas_cost_usdc", "rebalancing_cost_usdc",
        "net_profit_usdc", "break_even_edge_per_eth_usdc",
    ]].describe(percentiles=[.1, .5, .9]).T)

    cost_means = analysis[["exchange_fee_usdc", "gas_cost_usdc", "rebalancing_cost_usdc"]].mean()
    cost_means.plot(kind="bar", figsize=(8, 3), title="Mean cost per selected opportunity")
    plt.ylabel("USDC"); plt.xticks(rotation=20); plt.show()

    axes = analysis.plot.scatter(
        x="quantity_eth", y="break_even_edge_per_eth_usdc", c="gas_cost_usdc",
        colormap="viridis", figsize=(9, 4), title="Required price edge per ETH after modeled costs",
    )
    axes.set(xlabel="ETH quantity", ylabel="Break-even CEX–DEX difference (USDC/ETH)")
    plt.show()

## Adaptive optimizer diagnostics

In [ ]:
feasible = candidates[candidates["feasible"].eq(1)].copy()
if feasible.empty:
    print("No feasible candidate rows to inspect.")
else:
    size_profile = feasible.groupby(["direction", "quantity_eth"], as_index=False).agg(
        samples=("id", "size"), median_profit=("net_profit_usdc", "median"),
        profitable_share=("net_profit_usdc", lambda values: values.gt(0).mean()),
    )
    for direction, group in size_profile.groupby("direction"):
        plt.plot(group["quantity_eth"], group["median_profit"], marker=".", label=direction)
    plt.axhline(0, color="black", linewidth=1)
    plt.xscale("log")
    plt.xlabel("Quoted quantity (ETH, log scale)")
    plt.ylabel("Median net profit (USDC)")
    plt.title("Quoted sizing surface across stored observations")
    plt.legend(); plt.show()

    display(size_profile.sort_values(["direction", "median_profit"], ascending=[True, False]).groupby("direction").head(10))

## Paper-trading ledger

In [ ]:
if paper_trades.empty:
    print("No paper trades recorded. Run `python -m scr --config config.toml paper` to populate this section.")
else:
    for column in ["quantity_eth", "estimated_profit_usdc", "cumulative_pnl_usdc"]:
        paper_trades[column] = pd.to_numeric(paper_trades[column], errors="coerce")
    paper_trades["executed_at"] = pd.to_datetime(paper_trades["executed_at"], utc=True, errors="coerce")

    session_summary = paper_trades.groupby("session_id").agg(
        trades=("id", "size"),
        started_at=("executed_at", "min"),
        ended_at=("executed_at", "max"),
        final_cumulative_pnl=("cumulative_pnl_usdc", "last"),
        marked_trade_profit=("estimated_profit_usdc", "sum"),
    ).sort_values("ended_at", ascending=False)
    display(session_summary)

    for session_id, group in paper_trades.groupby("session_id"):
        plt.plot(group["executed_at"], group["cumulative_pnl_usdc"], marker=".", label=session_id[:8])
    plt.xlabel("Time"); plt.ylabel("Cumulative paper P&L (USDC)")
    plt.title("Paper P&L by session"); plt.legend(title="Session"); plt.show()

    latest_inventory = json.loads(paper_trades.iloc[-1]["inventory_json"])
    display(pd.Series(latest_inventory, name="latest stored inventory"))

## Interpretation checklist

- Prefer net profit and break-even edge over the raw CEX–DEX price difference.
- Treat repeated adjacent marks as one market episode, not independent trades.
- Exclude stale/error observations before drawing conclusions.
- Check latency survival before assuming a detected mark could have been captured.
- Paper P&L uses hypothetical immediate fills and configured costs; it is not exchange-account P&L.